In [3]:
# Extract_ABMC_Soldiers.py
# 5/13/2025

from bs4 import BeautifulSoup 
import datetime
from datetime import datetime as dt
from datetime import date 
from datetime import timedelta
import time
import re
from pandas import read_csv
import pandas as pd
# from selenium import webdriver
from seleniumwire import webdriver
from selenium.webdriver.common.by import By
import json
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
import sys
import random
from zenrows import ZenRowsClient
import requests
import math
import numpy as np
import urllib3
import inspect
# from IPython.display import display


# def maximize_screen():
#     from IPython.display import display, HTML
#     display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

def left(s, amount):
    return s[:amount]
def right(s, amount):
    return s[-amount:]
def mid(s, offset, amount):
    return s[offset:offset+amount]

def maximize_screen():
    from IPython.display import display, HTML
    display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

def timestamp():
    today=str(datetime.date.today()).split('-')
    year=today[0]
    month=today[1]
    day=today[2]
    a=str(datetime.datetime.now())
    time_split=re.split(':| ',a)
    hour=time_split[1]
    timestamp=year+month+day+hour
    return timestamp

def convert_date_z(date): 
    date_t= pd.to_datetime(date, utc=True)
    date_out=date_t.dt.strftime("%m/%d/%Y")
    return date_out

def convert_datetime_t(event_date_t):
    no_time='2000-01-01T00:00:00'
    no_time_dt=dt.strptime(no_time, "%Y-%m-%dT%H:%M:%S") 
    event_date_t = event_date_t.fillna(no_time)
    event_date_dt=pd.to_datetime(event_date_t, format="%Y-%m-%dT%H:%M:%S")
    event_date_index=event_date_dt.dt.strftime("%Y%m%d")
    event_date =event_date_dt.dt.strftime("%m/%d/%Y")
    event_date_rnd_dt = event_date_dt + timedelta(minutes=30)
    event_time_index=event_date_rnd_dt.dt.strftime('%H')  
    event_time = event_date_dt.dt.strftime('%H:%M')
    return     event_date,event_date_index,event_time,event_time_index 



def random_sleep(call_count): 
    global sleep_time0,listings_df
    if call_count%10==0:
        sleep_time =sleep_time0
        now_date = datetime.datetime.now() + timedelta(seconds=sleep_time)
        sleep_end_date=now_date.strftime("%m/%d/%Y %H:%M:%S")    
        print(f"\t\t\tSleeping for {sleep_time:.0f} seconds • Sleeping till: {sleep_end_date}")
    else:
        sleep_time = random.uniform(15, 30)
    return sleep_time 

def get_total_cemeteries(url):
    global my_header
    response = requests.request("GET", url, headers=my_header) 
    status_code=response.status_code
    url_source = response.text 
    
    beautifulSoupText = BeautifulSoup(url_source, 'html.parser') 
    specific_tag_text = beautifulSoupText.get_text()
    capture_json=json.loads(specific_tag_text) 
    total_cemeteries=capture_json['data']['cemeteries']['total']
    return total_cemeteries

def get_grave_count(url):
    global status_code
    capture_json=[]
    grave_count=0
    response = requests.request("GET", url, headers=my_header) 
    status_code=response.status_code
    url_source = response.text 
    if status_code!=200:print('\tstatus_code:',status_code, 'url',url);return capture_json,grave_count
    beautifulSoupText = BeautifulSoup(url_source, 'html.parser') 
    specific_tag_text = beautifulSoupText.get_text()
    capture_json=json.loads(specific_tag_text) 
    grave_count=capture_json['csw:GetRecordsResponse']['csw:SearchResults']['numberOfRecordsMatched']
    return capture_json,grave_count
    
def extract_abmc_grave_data(capture_json):
    global field_list_set,field_list
    collection_df=pd.DataFrame(capture_json['csw:GetRecordsResponse']['csw:SearchResults']['iStoreRecord'])
    collection_columns_set=set(list(collection_df))
    missing_columns=field_list_set-collection_columns_set
    collection_df[list(missing_columns)]=np.nan
    grave_frame_df=collection_df[field_list]
    return grave_frame_df

def clean_dataframe(graves_df):
    global field_list
    for column in field_list:
        try:
            graves_df[column]= graves_df[column].astype(str).replace(r"\,", ' * ', regex=True).str.strip()
            graves_df[column]= graves_df[column].astype(str).replace(r"\'", '', regex=True).str.strip()
        except:
            continue
    delete_list=[r'\"',r"\'",r'\?',r'\[',r'\]']
    for item in delete_list:
        for column in field_list:
            try:
                graves_df[column]= graves_df[column].astype(str).replace(item, ' ', regex=True).str.strip()
            except:
                continue
    return graves_df

#*************** Append  *******************************
append=0
# append_flag=input('To Append Press 1')
# if append_flag=='1':append=1
#*************** Append  *******************************

my_header = {
  'priority': 'u=1, i',
  'sec-ch-ua': '"Google Chrome";v="135", "Not-A.Brand";v="8", "Chromium";v="135"',
  'sec-ch-ua-platform': '"Windows"',
  'sec-fetch-mode': 'cors',
  'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/135.0.0.0 Safari/537.36',
  'Accept': 'application/json, text/plain, */*',
  'Accept-language': 'en-GB,en-US;q=0.9,en;q=0.8,ja;q=0.7',
  'cookie': 'XSRF-TOKEN=3ac46b49-9b6d-4fa0-82bf-b5dba745581f; SESSION=N2FhZmI5M2YtZTIxOC00YjFhLTgyNTctODc5MDhmM2MxYzBh',
  'referer': 'https://weremember.abmc.gov/',
  'x-requested-with': 'XMLHttpRequest',
    'x-xsrf-token': 'ffb7e355-ed35-4b99-95ce-8d7b02c1792a'
}
maximize_screen()
timestamp=timestamp()
collection_date0=datetime.datetime.now().strftime("%m/%d/%Y")
collection_date_dt=pd.to_datetime(collection_date0)
collection_date=collection_date0

start_time=datetime.datetime.now()
event_time=start_time

process_name='Extract ABMC_Graves '
print('**Starting *** ',process_name,' *** , Time:',start_time,' **\n')
DirMain="C:\\Users\\jrozm\\OneDrive\\Benjamin\\"



# fileIn="JewishCemeteries_CountyCorrelated"
fileOutput="ABMC_Graves"

# if append==1:fileIn='JewishCemeteries_CountyCorrelated_X'


# fileCemeteriesIn=DirMain+"OutputData\\Cemeteries\\"+fileIn+".csv"
fileOut=DirMain+"OutputData\\ABMC\\"+fileOutput+"_"+timestamp+".csv"

fileOut0=DirMain+"OutputData\\ABMC\\"+fileOutput+".csv"
fileOutInt=DirMain+"OutputData\\ABMC\\"+fileOutput+"_Interim.csv"


field_list=['war','title','first_name','middle_name','id','service_number','entry_state','branch','death_date','military_rank','aw_rank','unit','division','missing_status',\
 'country_name','cemetery','access_level','library','level_code', 'aw_library_id','lb_document_id','document_path','album_data','album_id','suffix']
field_list_set=set(field_list)
if append==1:print('**** Appending ****')    
if append!=1:graves_df=pd.DataFrame([])  

sleep_time0=3*60
graves_df=pd.DataFrame([])
get_counter=0
frame_count=0

grave_frame_count=1
if append==1:grave_frame_count=25001
grave_count=500000
max_records=500
url0 = "https://weremember.abmc.gov/aw-server/awcsw?CONSTRAINT=qs%3D%5B*%5D%2CqueryType%3D%5B16%5D%2Clibrary%3D%5BABMCBurialLib%20OR%20Reconciliation_Differences%5D&typenames=aw:iStoreRecord&OUTPUTFORMAT=application/json&SERVICE=CSW&REQUEST=GetRecords&CONSTRAINTLANGUAGE=AWQL_FORM&VERSION=2.0.2&ELEMENTSETNAME=summary&RESULTTYPE=results&STARTPOSITION="
url1 ="&MAXRECORDS="+str(max_records)+"&SORTBY=title:ASC"



# https://weremember.abmc.gov/aw-server/awcsw?CONSTRAINT=qs%3D%5B*%5D%2CqueryType%3D%5B16%5D%2Clibrary%3D%5BABMCBurialLib%20OR%20Reconciliation_Differences%5D%2Ccemetery%3D%5B%22Aisne-Marne%20American%20Cemetery%22%5D&typenames=aw:iStoreRecord&OUTPUTFORMAT=application/json&SERVICE=CSW&REQUEST=GetRecords&CONSTRAINTLANGUAGE=AWQL_FORM&VERSION=2.0.2&ELEMENTSETNAME=summary&RESULTTYPE=results&STARTPOSITION=151&MAXRECORDS=50&SORTBY=title:ASC&INVOKER=SYSTEM





while grave_frame_count<grave_count:
    url=url0+str(grave_frame_count)+url1
    capture_json,grave_count=get_grave_count(url)
    # print(url)
    print(json.dumps(capture_json))            
    input('capture_json')  
    if status_code!=200:print('\tstatus_code:',status_code);break 

    grave_frame_df=extract_abmc_grave_data(capture_json)
    grave_frame_df['death_date'] = pd.to_datetime(grave_frame_df['death_date'], format="%m/%d/%Y %H:%M:%S", errors='coerce')
    grave_frame_df['death_date']=grave_frame_df['death_date'].dt.strftime("%m/%d/%Y")
    graves_df = graves_df._append(grave_frame_df, ignore_index = True) 
    # print(graves_df)
    # input('graves_df')
    
    grave_frame_count+=len(grave_frame_df)
    graves_df=clean_dataframe(graves_df)
    graves_df.to_csv(fileOutInt,mode="w", index=False,header=True)
    print(f"    --- {process_name:<20}•  grave count: {grave_frame_count}/{grave_count} ")
    get_counter+=1
    sleep_time=random_sleep(get_counter)       
    time.sleep(sleep_time) 
       
end_time=datetime.datetime.now()
processing_time=end_time-start_time
processing_time_seconds=int(processing_time.total_seconds())
print(f"**** {process_name}   Processing Time: {processing_time_seconds} Sec")   
graves_df=clean_dataframe(graves_df)  
graves_df.to_csv(fileOut,mode="w", index=False,header=True)
graves_df.to_csv(fileOut0,mode="w", index=False,header=True) 
print("\n")
print('Uploaded To: ',fileOut0)



**Starting ***  Extract ABMC_Graves   *** , Time: 2025-08-01 12:58:45.989866  **

	status_code: 401 url https://weremember.abmc.gov/aw-server/awcsw?CONSTRAINT=qs%3D%5B*%5D%2CqueryType%3D%5B16%5D%2Clibrary%3D%5BABMCBurialLib%20OR%20Reconciliation_Differences%5D&typenames=aw:iStoreRecord&OUTPUTFORMAT=application/json&SERVICE=CSW&REQUEST=GetRecords&CONSTRAINTLANGUAGE=AWQL_FORM&VERSION=2.0.2&ELEMENTSETNAME=summary&RESULTTYPE=results&STARTPOSITION=1&MAXRECORDS=500&SORTBY=title:ASC
[]


KeyboardInterrupt: Interrupted by user

In [21]:
 # sleep_end_date=now_date.strftime("%m/%d/%Y %H:%M:%S") 
# death_date0 = dt.strptime(death_date, "%m/%d/%Y %H:%M:%S")
# death_date0
# death_date0 = dt.strptime(grave_frame_df['death_date'], "%m/%d/%Y %H:%M:%S")
grave_frame_df['death_date'] = pd.to_datetime(grave_frame_df['death_date'], format="%m/%d/%Y %H:%M:%S", errors='coerce')

In [35]:
capture_json

{'csw:GetRecordsResponse': {'xmlns:gml': 'http://www.opengis.net/gml',
  'xmlns:csw': 'http://www.opengis.net/cat/csw/2.0.2',
  'csw:SearchResults': {'expires': '2025-05-13T15:48:30Z',
   'xmlns:xsi': 'http://www.w3.org/2001/XMLSchema-instance',
   'numberOfRecordsMatched': 224542,
   'facetFields': [{'name': 'cemetery',
     'value': ['Aisne-Marne American Cemetery[3345]',
      'Ardennes American Cemetery[5773]',
      'Brittany American Cemetery[4908]',
      'Brookwood American Cemetery[1032]',
      'Cambridge American Cemetery[8936]',
      'Clark Veterans Cemetery[1]',
      'Corozal American Cemetery[5592]',
      'East Coast Memorial[4598]',
      'Epinal American Cemetery[5676]',
      'Flanders Field American Cemetery[411]',
      'Florence American Cemetery[5807]',
      'Henri-Chapelle American Cemetery[8430]',
      'Honolulu Memorial[28805]',
      'Lorraine American Cemetery[10926]',
      'Luxembourg American Cemetery[5445]',
      'Manila American Cemetery[53400]',
  